In [1]:
from google import genai
from google.genai import types
from PIL import Image

In [2]:
# 1. Initialize the client (automatically picks up GEMINI_API_KEY from environment)
client = genai.Client(
    vertexai=True,
    project="infinitas-ds-ai-poc",
    location="us-central1",
)

In [3]:
# 2. Open your local image file

paths = ["test_images/First_Tamper.png", "test_images/first-tamper.png", "test_images/combined.png","test_images/neat.png", "test_images/neatest.png"]
images = []
for path in paths:
    images.append(Image.open(path))

print(len(images))

5


In [4]:
# Read the V1 prompt

with open("prompt-library/V1.txt", "r", encoding="utf-8") as file:
    file_content = file.read()

prompt_v1 = file_content
print(type(prompt_v1))

# Read the V1 prompt split into system instruction + task prompt

with open("prompt-library/V1_system.txt", "r", encoding="utf-8") as file:
    system_prompt_v1 = file.read()

with open("prompt-library/V1_task.txt", "r", encoding="utf-8") as file:
    task_prompt_v1 = file.read()

<class 'str'>


## Try the first two image together

In [6]:
image_list = images[0:2] # image 1 and 2

for image in image_list:

    # 3. Generate content by passing both the image object and your text prompt
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            prompt_v1,
            image
        ], config = types.GenerateContentConfig(
            temperature = 0.1
        )
    )

    # 4. Print the text result
    print(response.text)

```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document displays a consistent typography throughout. All text, including Thai characters and numbers, maintains uniform font weight, sharpness, resolution, and color. There are no visible inconsistencies in font size, shadows, edges, or background around specific text areas. The baseline of the text is also consistent across all lines. The overall appearance suggests a clean, digitally generated document without any signs of post-recapture alteration.",
  "signals_detected": [],
  "notes": "The image quality is high, allowing for clear inspection of text details."
}
```
```json
{
  "verdict": "tampered",
  "confidence": "high",
  "reasoning": "The document's overall typography, including font style, weight, and sharpness, appears consistent across most text regions. However, a significant anomaly is present in the date line. The text 'พฤศจิกายน ๒๕๖๙' is enclosed within a distinct gray rectangular backgroun

In [7]:
# V1 split: persona passed as system_instruction, task prompt as the user turn

image_list = images[0:2] # image 1 and 2

for image in image_list:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[task_prompt_v1, image],
        config=types.GenerateContentConfig(
            system_instruction=system_prompt_v1,
            temperature = 0.1
        ),
    )
    print(response.text)

```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document displays consistent typography throughout. All text, including Thai script and numerals, exhibits uniform font weight, sharpness, and resolution. There are no discernible font size mismatches within text blocks (the title is appropriately larger, which is standard formatting). No anomalies such as mismatched shadows, irregular edges, or color/contrast discontinuities around specific characters or numbers were detected. The overall presentation suggests a single, coherent digital generation without any signs of post-recapture alteration.",
  "signals_detected": [],
  "notes": ""
}
```
```json
{
  "verdict": "tampered",
  "confidence": "high",
  "reasoning": "The text 'พฤศจิกายน ๒๕๖๙' is enclosed within a distinct gray rectangular box. This gray background is inconsistent with the white background of the rest of the document, creating a clear color and contrast discontinuity isolated to this specific

### Try with other images
- separated for ease of reading

In [8]:
# Other images
image_list = images[2:6] #image 3,4,5

for image in image_list:

    # 3. Generate content by passing both the image object and your text prompt
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            prompt_v1,
            image
        ], config = types.GenerateContentConfig(
            temperature = 0.1)
    )

    # 4. Print the text result
    print(response.text)

```json
{
  "verdict": "tampered",
  "confidence": "high",
  "reasoning": "The document exhibits clear visual signals of digital alteration in two distinct text regions. Both the date portion ('พฤศจิกายน ๒๕๖๙') and the numerical amount ('๖๐๐,๐๐๐') are presented on uniform grey rectangular backgrounds. These grey backgrounds have unnaturally sharp and clean edges, suggesting they are digitally inserted overlays rather than part of the original document's rendering or a natural highlight. Furthermore, the text characters within these grey boxes appear to have slightly lower resolution and softer edges compared to the surrounding, unaltered text, indicating that these specific text elements, along with their backgrounds, were digitally inserted or modified after the document's initial generation or capture.",
  "signals_detected": [
    {
      "signal_type": "shadow_edge_anomaly",
      "location": "Date field: 'พฤศจิกายน ๒๕๖๙'",
      "severity": "high"
    },
    {
      "signal_type":

In [9]:
# Other images with separate prompts
image_list = images[2:6] #image 3,4,5

for image in image_list:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[task_prompt_v1, image],
        config=types.GenerateContentConfig(
            system_instruction=system_prompt_v1,
            temperature = 0.1
        ),
    )
    print(response.text)

```json
{
  "verdict": "tampered",
  "confidence": "high",
  "reasoning": "The document exhibits a consistent typography throughout most of its content. However, two specific text regions show clear signs of post-recapture digital alteration. The date '๙ พฤศจิกายน ๒๕๖๙' and the amount '๖๐๐,๐๐๐' are both enclosed within solid gray rectangular boxes. These gray boxes are visually distinct from the document's original clean white background, indicating a 'mismatched background isolated to specific text areas'. Furthermore, upon close examination, the text characters within these gray boxes appear slightly less sharp and have a subtly different rendering quality compared to the surrounding text (e.g., 'วันที่' or 'จำนวนเงินกู้'). This suggests an inconsistency in 'sharpness or resolution relative to surrounding text', likely resulting from the text being re-rendered or flattened along with the digitally inserted gray background.",
  "signals_detected": [
    {
      "signal_type": "shadow_

In [ ]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      avg_logprobs=-2.073611407212808,
      citation_metadata=CitationMetadata(
        citations=[
          Citation(
            end_index=2479,
            start_index=2230,
            uri='https://www3.ago.go.th/center/wp-content/uploads/2022/06/Ktb.doc'
          ),
          Citation(
            end_index=2604,
            start_index=2392,
            uri='https://capr.tsu.ac.th/UserFiles/04-%E0%B8%AB%E0%B8%99%E0%B8%B1%E0%B8%87%E0%B8%AA%E0%B8%B7%E0%B8%AD%E0%B8%A3%E0%B8%B1%E0%B8%9A%E0%B8%A3%E0%B8%AD%E0%B8%87%E0%B9%80%E0%B8%A3%E0%B8%B7%E0%B9%88%E0%B8%AD%E0%B8%87%E0%B8%81%E0%B8%B2%E0%B8%A3%E0%B8%AB%E0%B8%B1%E0%B8%81%E0%B9%80%E0%B8%87%E0%B8%B4%E0%B8%99%E0%B9%80%E0%B8%94%E0%B8%B7%E0%B8%AD%E0%B8%99.doc'
          ),
          Citation(
            end_index=2761,
            start_index=2503,
            uri='https://capr.tsu.ac.th/UserFiles/04-%E0%B8%AB%E0%B8%99%E0%B8%B1%E0%B8%87%E0%B8